# ♟️ Modular Chess Engine V1.5 — Дообучение (Fine-Tuning) на позициях после середины игры и эндшпиле
Этот блокнот предназначен для **целевого дообучения (Fine-Tuning)** нейросетевого шахматного движка **ModularChessNet-V1.5** (6.58M параметров, SE-ResNet) на позициях **позднего миттельшпиля и эндшпиля** на бесплатном GPU **NVIDIA Tesla T4** в Google Colab.

### 🎯 Зачем нужно дообучение на поздней стадии игры?
1. **Дисбаланс датасетов**: В стандартных базах партий ~75% позиций относятся к дебюту и раннему миттельшпилю. До глубокого эндшпиля партии доходят реже, из-за чего базовые сети могут терять инициативу, топтаться фигурами и упускать форсированные выигрыши.
2. **Проведение пешек**: Модель учится форсированно толкать проходные пешки в ферзи и точно рассчитывать темпы пешечных прорывов.
3. **Техника реализации перевеса**: Чёткий отбор позиций от Stockfish обучает сеть безошибочной технике реализации лишнего качества, ладейным окончаниям и оппозиции королей.
4. **Повышенный вес Value Head**: В эндшпиле точная оценка (+1.0 / -1.0 / 0.0) критичнее дебютных полутонов, поэтому вес функции ценности оптимизирован для решающих стадий.

### ⚙️ Рекомендуемая среда:
> **Среда выполнения (Runtime)** ➔ **Сменить среду (Change runtime type)** ➔ **T4 GPU** ➔ **Сохранить**.

In [ ]:
# 1. Проверка доступности GPU Tesla T4 и тензорных ядер (Tensor Cores)
import torch

print(f"PyTorch Version: {torch.__version__}")
if not torch.cuda.is_available():
    raise SystemError("⚠️ GPU не обнаружен! Включите T4 GPU в меню: Среда выполнения -> Сменить среду -> T4 GPU.")

device = torch.device("cuda")
gpu_name = torch.cuda.get_device_name(0)
capability = torch.cuda.get_device_capability(0)

print(f"✅ Активный GPU: {gpu_name}")
print(f"⚡ Compute Capability: {capability[0]}.{capability[1]}")
if capability[0] >= 7:
    print("🔥 Тензорные ядра (Tensor Cores Turing) активны! Поддерживается аппаратный FP16 AMP.")

torch.backends.cudnn.benchmark = True

In [ ]:
# 2. Установка необходимых библиотек
!pip install -q python-chess pyarrow datasets tqdm onnx matplotlib
print("✅ Зависимости успешно установлены!")

In [ ]:
# 3. Архитектура нейросети ModularChessNet-V1.5 (SE-ResNet)
import torch.nn as nn

class SEBlock(nn.Module):
    def __init__(self, channels=192, reduction=4):
        super().__init__()
        self.fc1 = nn.Linear(channels, channels // reduction, bias=False)
        self.fc2 = nn.Linear(channels // reduction, channels, bias=False)
        self.relu = nn.ReLU(inplace=True)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        b, c, _, _ = x.size()
        y = x.mean((2, 3))
        y = self.relu(self.fc1(y))
        y = self.sigmoid(self.fc2(y)).view(b, c, 1, 1)
        return x * y

class ResBlockSE(nn.Module):
    def __init__(self, channels=192):
        super().__init__()
        self.conv1 = nn.Conv2d(channels, channels, 3, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(channels)
        self.conv2 = nn.Conv2d(channels, channels, 3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(channels)
        self.se = SEBlock(channels, reduction=4)
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        res = x
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out = self.se(out)
        return self.relu(out + res)

class ModularChessNetV15(nn.Module):
    def __init__(self, num_res_blocks=8, channels=192):
        super().__init__()
        self.in_conv = nn.Conv2d(16, channels, kernel_size=3, padding=1, bias=False)
        self.in_bn = nn.BatchNorm2d(channels)
        self.relu = nn.ReLU(inplace=True)

        self.blocks = nn.ModuleList([ResBlockSE(channels) for _ in range(num_res_blocks)])

        # Policy Head (4x8x8 = 256 -> 4096 возможных ходов)
        self.policy_conv = nn.Conv2d(channels, 4, kernel_size=1, bias=False)
        self.policy_bn = nn.BatchNorm2d(4)
        self.policy_fc = nn.Linear(4 * 8 * 8, 4096)

        # Value Head (2x8x8 = 128 -> 256 -> 1)
        self.value_conv = nn.Conv2d(channels, 2, kernel_size=1, bias=False)
        self.value_bn = nn.BatchNorm2d(2)
        self.value_fc1 = nn.Linear(2 * 8 * 8, 256)
        self.value_fc2 = nn.Linear(256, 1)

    def forward(self, x):
        out = self.relu(self.in_bn(self.in_conv(x)))
        for block in self.blocks:
            out = block(out)

        # Policy
        p = self.relu(self.policy_bn(self.policy_conv(out)))
        policy_logits = self.policy_fc(p.flatten(1))

        # Value
        v = self.relu(self.value_bn(self.value_conv(out)))
        v = self.relu(self.value_fc1(v.flatten(1)))
        value = torch.tanh(self.value_fc2(v))

        return policy_logits, value

print("✅ Архитектура ModularChessNet-V1.5 успешно определена.")

In [ ]:
# 4. Загрузка исходных весов V1.5 для дообучения (Checkpoint Loader)
import os

model = ModularChessNetV15(num_res_blocks=8, channels=192).to(device)
param_count = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Параметров модели: {param_count:,} (~{param_count/1e6:.2f}M)")

# Поиск файла весов в текущей директории
checkpoint_candidates = ["modular_chess_v1_5.pt", "best_model V1.5.pt", "modular_chess_v1.pt"]
loaded_checkpoint = None

for cp in checkpoint_candidates:
    if os.path.exists(cp):
        loaded_checkpoint = cp
        break

if loaded_checkpoint:
    print(f"📦 Найден локальный чекпоинт: '{loaded_checkpoint}'! Загрузка весов...")
    state_dict = torch.load(loaded_checkpoint, map_location=device)
    model.load_state_dict(state_dict, strict=True)
    print("✅ Исходные веса успешно загружены в модель!")
else:
    print("ℹ️ Файл весов не найден локально.")
    print("Вы можете загрузить 'modular_chess_v1_5.pt' через левую панель Colab (вкладка Файлы -> Загрузить) или с помощью ячейки ниже.")

In [ ]:
# (Опционально) Загрузка файла весов через диалоговое окно Colab
from google.colab import files

if not loaded_checkpoint:
    print("Выберите файл 'modular_chess_v1_5.pt' или 'best_model V1.5.pt' на компьютере:")
    uploaded = files.upload()
    for fn in uploaded.keys():
        if fn.endswith('.pt'):
            state_dict = torch.load(fn, map_location=device)
            model.load_state_dict(state_dict, strict=True)
            print(f"✅ Веса из '{fn}' успешно загружены в модель!")
            loaded_checkpoint = fn
            break

In [ ]:
# 5. Скачивание датасета Stockfish Evaluations (HuggingFace)
import urllib.request

dataset_files = [
    ("stockfish_evals_part0.parquet", "https://huggingface.co/datasets/Lichess/chess-position-evaluations/resolve/refs%2Fconvert%2Fparquet/default/train/0000.parquet"),
    ("stockfish_evals_part1.parquet", "https://huggingface.co/datasets/Lichess/chess-position-evaluations/resolve/refs%2Fconvert%2Fparquet/default/train/0001.parquet")
]

for fname, url in dataset_files:
    if not os.path.exists(fname):
        print(f"📥 Скачивание {fname} с HuggingFace (скорость сети Colab ~150-250 МБ/с)...")
        urllib.request.urlretrieve(url, fname)
        print(f"✅ {fname} успешно загружен!")
    else:
        print(f"ℹ️ {fname} уже готов на диске.")

In [ ]:
# 6. Интеллектуальная фильтрация: отбор позиций после середины игры и эндшпиля
import pyarrow.parquet as pq
import numpy as np
from tqdm.auto import tqdm

# Целевой объём позиций для дообучения (500 000 - 800 000 позиций идеально для fine-tuning)
TARGET_SAMPLES = 600_000

PIECE_MAP = {
    'P': 0, 'N': 1, 'B': 2, 'R': 3, 'Q': 4, 'K': 5,
    'p': 6, 'n': 7, 'b': 8, 'r': 9, 'q': 10, 'k': 11
}

VAL_MAP = {'p': 1, 'n': 3, 'b': 3, 'r': 5, 'q': 9}

def is_late_game(fen: str) -> bool:
    """
    Определяет, относится ли позиция к стадии после середины игры или эндшпилю:
    1. Число фигур на доске <= 18 (включая королей), ИЛИ
    2. Материал без учета королей <= 40 очков (менее половины от стартового), ИЛИ
    3. Глубокий ход партии (ход 25+) при сократившемся числе фигур (<= 24 фигуры).
    """
    parts = fen.split()
    board_str = parts[0]
    pieces = [ch for ch in board_str if ch.lower() in 'pnbrqk']
    total_pieces = len(pieces)
    
    # Критерий 1: Резко сократившееся число фигур
    if total_pieces <= 18:
        return True
        
    # Критерий 2: Суммарный материал сторон
    mat_score = sum(VAL_MAP.get(ch.lower(), 0) for ch in board_str)
    if mat_score <= 40:
        return True
        
    # Критерий 3: Глубокая стадия партии (ход 25+)
    fullmove = int(parts[5]) if len(parts) > 5 and parts[5].isdigit() else 1
    if fullmove >= 25 and total_pieces <= 24:
        return True
        
    return False

def fen_to_tensor_fast(fen: str) -> np.ndarray:
    planes = np.zeros((16, 8, 8), dtype=np.uint8)
    parts = fen.split()
    board_str = parts[0]
    turn_str = parts[1] if len(parts) > 1 else 'w'
    castling_str = parts[2] if len(parts) > 2 else '-'
    
    r, c = 0, 0
    for ch in board_str:
        if ch == '/':
            r += 1
            c = 0
        elif ch.isdigit():
            c += int(ch)
        else:
            p_idx = PIECE_MAP.get(ch)
            if p_idx is not None:
                planes[p_idx, r, c] = 1
            c += 1
            
    if turn_str == 'w':
        planes[12, :, :] = 1
    if 'K' in castling_str: planes[13, 0:4, 0:4] = 1
    if 'Q' in castling_str: planes[13, 0:4, 4:8] = 1
    if 'k' in castling_str: planes[14, 4:8, 0:4] = 1
    if 'q' in castling_str: planes[14, 4:8, 4:8] = 1
    return planes

def uci_to_policy_idx(uci: str):
    if not uci or len(uci) < 4:
        return None
    from_c = ord(uci[0]) - 97
    from_r = 8 - int(uci[1])
    to_c = ord(uci[2]) - 97
    to_r = 8 - int(uci[3])
    if 0 <= from_r < 8 and 0 <= from_c < 8 and 0 <= to_r < 8 and 0 <= to_c < 8:
        return (from_r * 8 + from_c) * 64 + (to_r * 8 + to_c)
    return None

print(f"🔍 Фильтрация и извлечение {TARGET_SAMPLES:,} эндшпильных позиций в память (uint8)...")
all_boards = np.zeros((TARGET_SAMPLES, 16, 8, 8), dtype=np.uint8)
all_policies = np.zeros(TARGET_SAMPLES, dtype=np.int64)
all_values = np.zeros((TARGET_SAMPLES, 1), dtype=np.float32)

loaded = 0
stats = {'pawn_endgame': 0, 'rook_endgame': 0, 'queenless': 0, 'late_mid': 0}
pbar = tqdm(total=TARGET_SAMPLES, desc="Фильтрация позиций")

for fname, _ in dataset_files:
    if loaded >= TARGET_SAMPLES:
        break
    parquet_file = pq.ParquetFile(fname)
    for batch in parquet_file.iter_batches(batch_size=32768, columns=['fen', 'line', 'cp', 'mate']):
        df = batch.to_pandas()
        for _, row in df.iterrows():
            fen = row['fen']
            if not is_late_game(fen):
                continue
                
            line = row['line']
            if not line:
                continue
            pol_idx = uci_to_policy_idx(line.split()[0])
            if pol_idx is None:
                continue
                
            cp = row['cp']
            mate = row['mate']
            if mate is not None and not np.isnan(mate):
                val = 1.0 if mate > 0 else -1.0
            elif cp is not None and not np.isnan(cp):
                val = float(np.tanh(cp / 400.0))
            else:
                continue
                
            all_boards[loaded] = fen_to_tensor_fast(fen)
            all_policies[loaded] = pol_idx
            all_values[loaded, 0] = val
            
            # Статистика типов окончаний
            board_part = fen.split()[0]
            has_rooks = 'r' in board_part or 'R' in board_part
            has_queens = 'q' in board_part or 'Q' in board_part
            has_minors = any(p in board_part for p in 'nbNB')
            if not has_rooks and not has_queens and not has_minors:
                stats['pawn_endgame'] += 1
            elif has_rooks and not has_queens and not has_minors:
                stats['rook_endgame'] += 1
            elif not has_queens:
                stats['queenless'] += 1
            else:
                stats['late_mid'] += 1
                
            loaded += 1
            pbar.update(1)
            if loaded >= TARGET_SAMPLES:
                break
        if loaded >= TARGET_SAMPLES:
            break

pbar.close()
print(f"🎉 Успешно собрано {loaded:,} эндшпильных позиций!")
print(f"📊 Статистика выборки:")
print(f"   • Пешечные эндшпили (K+P vs K+P): {stats['pawn_endgame']:,}")
print(f"   • Чистые ладейные эндшпили (R+P): {stats['rook_endgame']:,}")
print(f"   • Безферзевые окончания (ладьи + лёгкие фигуры): {stats['queenless']:,}")
print(f"   • Поздний миттельшпиль / ферзевые окончания: {stats['late_mid']:,}")

In [ ]:
# 7. Создание DataLoader с Pinned Memory
from torch.utils.data import TensorDataset, DataLoader

X_tensor = torch.from_numpy(all_boards[:loaded])
y_policy = torch.from_numpy(all_policies[:loaded])
y_value = torch.from_numpy(all_values[:loaded])

del all_boards, all_policies, all_values

dataset = TensorDataset(X_tensor, y_policy, y_value)

# Оптимальный Batch Size для Tensor Cores Turing на T4
BATCH_SIZE = 1024

train_loader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    pin_memory=True,
    num_workers=2,
    persistent_workers=True,
    drop_last=True
)

print(f"✅ DataLoader готов! Пакетов (батчей) на эпоху: {len(train_loader):,} (размер пакета: {BATCH_SIZE})")

In [ ]:
# 8. Запуск Fine-Tuning (Дообучения) на GPU Tesla T4 (FP16 AMP)
import time

EPOCHS = 8
# Пониженный learning rate для деликатного дообучения без разрушения дебютных знаний
FINE_TUNE_LR = 2e-4

optimizer = torch.optim.AdamW(model.parameters(), lr=FINE_TUNE_LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)

policy_loss_fn = nn.CrossEntropyLoss()
value_loss_fn = nn.MSELoss()
scaler = torch.cuda.amp.GradScaler()

history_loss = []
history_policy_loss = []
history_value_loss = []

print(f"🚀 Запуск {EPOCHS} эпох дообучения (LR: {FINE_TUNE_LR}, FP16 Tensor Cores, Value Weight: 0.8)...")
start_time = time.time()

for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss, total_pol, total_val = 0.0, 0.0, 0.0
    n_batches = len(train_loader)
    
    epoch_bar = tqdm(train_loader, desc=f"Эпоха {epoch:02d}/{EPOCHS}", leave=False)
    
    for b_uint8, target_policy, target_value in epoch_bar:
        # Преобразование uint8 -> float32 на лету в видеопамяти
        b = b_uint8.to(device, non_blocking=True).to(torch.float32)
        target_policy = target_policy.to(device, non_blocking=True)
        target_value = target_value.to(device, non_blocking=True)
        
        optimizer.zero_grad(set_to_none=True)
        
        with torch.cuda.amp.autocast(dtype=torch.float16):
            pred_policy, pred_value = model(b)
            loss_pol = policy_loss_fn(pred_policy, target_policy)
            loss_val = value_loss_fn(pred_value, target_value)
            # В эндшпиле точная оценка позиции особенно важна (вес 0.8)
            loss = loss_pol + 0.8 * loss_val
            
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        
        total_loss += loss.item()
        total_pol += loss_pol.item()
        total_val += loss_val.item()
        
        epoch_bar.set_postfix({
            'loss': f"{loss.item():.4f}",
            'policy': f"{loss_pol.item():.4f}",
            'value': f"{loss_val.item():.4f}"
        })
        
    scheduler.step()
    avg_loss = total_loss / n_batches
    avg_pol = total_pol / n_batches
    avg_val = total_val / n_batches
    curr_lr = scheduler.get_last_lr()[0]
    
    history_loss.append(avg_loss)
    history_policy_loss.append(avg_pol)
    history_value_loss.append(avg_val)
    
    print(f"Эпоха {epoch:02d}/{EPOCHS} | Loss: {avg_loss:.4f} (Policy: {avg_pol:.4f}, Value MSE: {avg_val:.4f}) | LR: {curr_lr:.7f}")

elapsed_min = (time.time() - start_time) / 60
print(f"\n🏆 Дообучение успешно завершено за {elapsed_min:.2f} минут!")

In [ ]:
# 9. График сходимости функции потерь (Loss Curves)
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 4))
plt.plot(range(1, EPOCHS + 1), history_loss, label='Общий Loss (Policy + 0.8*Value)', color='#6366f1', lw=2.5)
plt.plot(range(1, EPOCHS + 1), history_policy_loss, label='Policy Loss (CE)', color='#10b981', ls='--', lw=2)
plt.plot(range(1, EPOCHS + 1), history_value_loss, label='Value Loss (MSE)', color='#f59e0b', ls=':', lw=2)
plt.xlabel('Эпоха')
plt.ylabel('Loss')
plt.title('Динамика сходимости при дообучении на эндшпильных позициях')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# 10. Тактическое тестирование дообученной модели на эндшпильных позициях
model.eval()

benchmark_positions = [
    ("8/5k2/4p3/3pP3/3P1P2/8/8/5K2 w - - 0 40", "Пешечный прорыв: f4-f5!", "f4f5"),
    ("8/8/4k3/4P3/8/8/8/4K3 w - - 0 50", "Оппозиция и проведение пешки: e5-e6 или Кр", ["e5e6", "e1e2", "e1f2"]),
    ("4k3/8/8/8/8/8/4R3/4K3 w - - 0 60", "Ладейный эндшпиль: Шах или отсечение", ["e2e8", "e2e7", "e2e4", "e2a2"]),
    ("8/1P6/8/8/8/5k2/8/4K3 w - - 0 55", "Превращение проходной пешки в ферзя: b7-b8!", "b7b8"),
    ("r1bqkbnr/pppppppp/8/8/4P3/8/PPPP1PPP/RNBQKBNR b KQkq e3 0 1", "Контрольный тест дебюта (1. e4): ответ c5/e5/e6", ["e7e5", "c7c5", "e7e6", "c7c6"])
]

print("🎯 Результаты тестирования дообученной модели:")
print("-" * 70)

for fen, desc, expected in benchmark_positions:
    inp = torch.from_numpy(fen_to_tensor_fast(fen)).unsqueeze(0).to(device).float()
    with torch.no_grad():
        pol_logits, val = model(inp)
        top_idx = torch.argmax(pol_logits, dim=-1).item()
        eval_val = val.item()
        
    f_sq, t_sq = divmod(top_idx, 64)
    f_r, f_c = divmod(f_sq, 8)
    t_r, t_c = divmod(t_sq, 8)
    pred_uci = chr(97 + f_c) + str(8 - f_r) + chr(97 + t_c) + str(8 - t_r)
    
    exp_str = ", ".join(expected) if isinstance(expected, list) else expected
    match = (pred_uci in expected) if isinstance(expected, list) else (pred_uci == expected)
    icon = "✅" if match else "ℹ️"
    
    print(f"{icon} Тест: {desc}")
    print(f"   Выбор модели: {pred_uci} | Оценка позиции (Value): {eval_val:+.3f} | Ожидалось: {exp_str}")
    print("-" * 70)

In [ ]:
# 11. Сохранение чекпоинта PyTorch и экспорт в формат ONNX
import onnx

# Сохранение PyTorch весов
torch.save(model.state_dict(), "modular_chess_v1_5_finetuned.pt")
print("💾 Сохранены PyTorch веса: modular_chess_v1_5_finetuned.pt")

# Экспорт в ONNX для прямого использования в веб-интерфейсе и сервере
model.eval()
dummy_input = torch.randn(1, 16, 8, 8, device=device)
onnx_filename = "modular_chess_v1_5.onnx"

torch.onnx.export(
    model,
    dummy_input,
    onnx_filename,
    input_names=["board_state"],
    output_names=["policy_logits", "value"],
    dynamic_axes={
        "board_state": {0: "batch_size"},
        "policy_logits": {0: "batch_size"},
        "value": {0: "batch_size"}
    },
    opset_version=14
)

# Проверка валидности ONNX графа
onnx_model = onnx.load(onnx_filename)
onnx.checker.check_model(onnx_model)
print(f"🚀 Модель успешно экспортирована в ONNX: '{onnx_filename}' (Opset 14, валидация пройдена)!")

# Автоматическое скачивание готового файла на компьютер
try:
    from google.colab import files
    print("📥 Запуск скачивания modular_chess_v1_5.onnx на ваш компьютер...")
    files.download(onnx_filename)
except Exception as e:
    print("ℹ️ Для скачивания перейдите во вкладку 'Файлы' слева и скачайте 'modular_chess_v1_5.onnx'.")